**Author:** Ashutosh Jayant  
**Project:** India State Competitiveness Index (ISCI)

# 04 - Index construction

Builds the final competitiveness score and ranking from the master table made in Notebook 03, and writes the results to the results folder.

## Setup
Load pandas and the master table from Notebook 03.

In [1]:
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent

master = pd.read_csv(root / "data" / "processed" / "master_features.csv", index_col=0)
print(master.shape)
print(master.columns.tolist())

(36, 14)
['ger', 'life_expectancy', 'unemp_rural', 'unemp_urban', 'cd_ratio', 'per_capita_power', 'td_losses', 'road_density', 'factory_density', 'msme_density', 'manufacturing_share', 'investment_rate', 'population', 'per_capita_nsdp']


## Choose the indicators
Average rural and urban unemployment into one number, then keep the 11 index indicators in their two groups.

In [2]:
INDEX_FEATURES = {
    "factor_conditions": [
        "ger", "life_expectancy", "unemployment_rate", "cd_ratio",
        "per_capita_power", "td_losses", "road_density",
    ],
    "related_supporting": [
        "factory_density", "msme_density", "manufacturing_share", "investment_rate",
    ],
}

master["unemployment_rate"] = master[["unemp_rural", "unemp_urban"]].mean(axis=1)

indicators = INDEX_FEATURES["factor_conditions"] + INDEX_FEATURES["related_supporting"]
analytical = master[indicators].copy()

# unemployment average assumes both series exist for a state; confirm coverage matches
both = master[["unemp_rural", "unemp_urban"]].notna().sum(axis=1)
print("states with both unemployment series:", (both == 2).sum())
print("states with only one:", (both == 1).sum())

print(analytical.shape)
print(analytical.notna().sum().to_string())

states with both unemployment series: 34
states with only one: 1
(36, 11)
ger                    36
life_expectancy        22
unemployment_rate      35
cd_ratio               36
per_capita_power       34
td_losses              33
road_density           33
factory_density        34
msme_density           34
manufacturing_share    34
investment_rate        34


## Mark direction
Note the two indicators where a higher value is worse (unemployment and T&D losses); the other nine are better when higher.

In [3]:
NEGATIVE = {"unemployment_rate", "td_losses"}

direction = pd.Series("positive", index=indicators, name="direction")
direction[list(NEGATIVE)] = "negative"

assert set(direction.index) == set(indicators), "direction and indicator set mismatch"
print(direction.to_string())

ger                    positive
life_expectancy        positive
unemployment_rate      negative
cd_ratio               positive
per_capita_power       positive
td_losses              negative
road_density           positive
factory_density        positive
msme_density           positive
manufacturing_share    positive
investment_rate        positive


## Scale to 0 to 1
Rescale every indicator to a 0 to 1 range, and flip the two marked ones so a high score always means good.

In [4]:
def normalize(col, is_negative):
    lo, hi = col.min(), col.max()
    if hi == lo:
        # every state has the same value; give a neutral 0.5 and keep missing as missing
        return col.where(col.isna(), 0.5)
    scaled = (col - lo) / (hi - lo)
    return 1 - scaled if is_negative else scaled

scores = pd.DataFrame({
    name: normalize(analytical[name], name in NEGATIVE)
    for name in indicators
})

print(scores.describe().round(3).loc[["min", "max"]].to_string())
print()
print(scores.round(3).head().to_string())

     ger  life_expectancy  unemployment_rate  cd_ratio  per_capita_power  td_losses  road_density  factory_density  msme_density  manufacturing_share  investment_rate
min  0.0              0.0                0.0       0.0               0.0        0.0           0.0              0.0           0.0                  0.0              0.0
max  1.0              1.0                1.0       1.0               1.0        1.0           1.0              1.0           1.0                  1.0              1.0

                             ger  life_expectancy  unemployment_rate  cd_ratio  per_capita_power  td_losses  road_density  factory_density  msme_density  manufacturing_share  investment_rate
entity                                                                                                                                                                                        
Andaman & Nicobar Islands  0.694              NaN                NaN     0.275               NaN      0.496         

## Check coverage
Count how many of the 11 indicators each state has, and mark which states clear the 8-of-11 rule.

In [5]:
available = scores.notna().sum(axis=1)
coverage_pct = (available / len(indicators) * 100).round().astype(int)
eligible = available >= 8

summary = pd.DataFrame({
    "Indicators_Available": available,
    "Coverage_Percent": coverage_pct,
    "Eligible": eligible,
})

print("eligible states:", int(eligible.sum()), "of", len(scores))
print()
print(summary.sort_values("Indicators_Available").to_string())

eligible states: 33 of 36

                                      Indicators_Available  Coverage_Percent  Eligible
entity                                                                                
Dadra & Nagar Haveli and Daman & Diu                     4                36     False
Lakshadweep                                              5                45     False
Ladakh                                                   7                64     False
Andaman & Nicobar Islands                                8                73      True
Jammu & Kashmir                                         10                91      True
Goa                                                     10                91      True
Chandigarh                                              10                91      True
Arunachal Pradesh                                       10                91      True
Nagaland                                                10                91      True
Puducherry      

## Save the indicator scores
Write the scaled indicators to results/indicator_scores.csv.

In [6]:
from pathlib import Path

results = root / "results"
results.mkdir(exist_ok=True)

scores.to_csv(results / "indicator_scores.csv", index_label="state")

print("Saved:", results / "indicator_scores.csv")
print("Shape:", scores.shape)

Saved: D:\india-state-competitiveness-index\results\indicator_scores.csv
Shape: (36, 11)


## Dimension scores
Average the indicators inside each group and save them as a diagnostic.

In [7]:
dimension_scores = pd.DataFrame({
    dim: scores[cols].mean(axis=1)
    for dim, cols in INDEX_FEATURES.items()
})

dimension_scores.to_csv(results / "dimension_scores.csv", index_label="state")

print(dimension_scores.round(3).head(10).to_string())

                                      factor_conditions  related_supporting
entity                                                                     
Andaman & Nicobar Islands                         0.395               0.112
Andhra Pradesh                                    0.529               0.467
Arunachal Pradesh                                 0.261               0.102
Assam                                             0.384               0.300
Bihar                                             0.294               0.161
Chandigarh                                        0.453               0.245
Chhattisgarh                                      0.318               0.364
Dadra & Nagar Haveli and Daman & Diu              0.707                 NaN
Delhi                                             0.565               0.236
Goa                                               0.467               0.825


## Final score and ranking
Average each eligible state's scores, rank the states, add the coverage columns, and save.

In [8]:
overall = scores.mean(axis=1)
overall[~eligible] = float("nan")

index = pd.DataFrame({
    "competitiveness_score": overall,
    "Rank": overall.rank(ascending=False, method="min").astype("Int64"),
    "Indicators_Available": available,
    "Coverage_Percent": coverage_pct,
}).sort_values("competitiveness_score", ascending=False)

index.to_csv(results / "competitiveness_index.csv", index_label="state")
print(index.to_string())

                                      competitiveness_score  Rank  Indicators_Available  Coverage_Percent
entity                                                                                                   
Goa                                                0.609971     1                    10                91
Tamil Nadu                                         0.604530     2                    11               100
Gujarat                                            0.594486     3                    11               100
Puducherry                                         0.565048     4                    10                91
Telangana                                          0.523265     5                    11               100
Andhra Pradesh                                     0.506662     6                    11               100
Punjab                                             0.505855     7                    11               100
Maharashtra                                   